# Blockwise attention with online softmax in lax.scan

FlashAttention never builds the full $T \times S$ score matrix. It walks over key blocks, keeping for each query row a running maximum $m$, a running normaliser $\ell$ and an unnormalised output $a$, and rescales them whenever the maximum grows. Write that algorithm with `jax.lax.scan` over key blocks.

Write `blockwise_attention(q, k, v, block, causal)`:

- `q`: `(T, d)`; `k`, `v`: `(S, d)`; float32. Scores are $s_{ij} = q_i \cdot k_j / \sqrt d$.
- `block`: Python int, the number of keys per block. `S` need not be a multiple of `block`: pad the keys and values up to the next multiple and mask the padding.
- `causal`: Python bool. If true, query $i$ may see key $j$ only when $j \le i + (S - T)$ (queries are the **last** $T$ positions of the sequence, as in decoding with a cache). `S >= T` when causal.

For each block, with $\tilde s$ the masked scores of that block:

$$m' = \max(m, \max_j \tilde s_j), \qquad \ell \leftarrow e^{m - m'} \ell + \sum_j e^{\tilde s_j - m'}, \qquad a \leftarrow e^{m - m'} a + \sum_j e^{\tilde s_j - m'} v_j, \qquad m \leftarrow m'$$

Return `(out, lse)`: `out = a / ℓ` with shape `(T, d)` and `lse = m + log ℓ` with shape `(T,)`, the log-sum-exp of each row's visible scores. Every row sees at least one key, but **a whole block can be masked for some rows**, so the update must not produce `nan` when $m$ and every $\tilde s_j$ are $-\infty$.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def blockwise_attention(q, k, v, block, causal):
    """Return (out (T, d), lse (T,)) computed one key block at a time."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [2.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0], [2.0, 2.0]], "dtype": "float64"}, 2, false], "e": {"__tup__": [{"__nd__": [[1.4359461069107056, 1.2919800281524658]], "dtype": "float32"}, {"__nd__": [1.9659039974212646], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float64"}, 1, true], "e": {"__tup__": [{"__nd__": [[1.0, 2.0], [2.3395230770111084, 3.3395230770111084]], "dtype": "float32"}, {"__nd__": [0.7071067690849304, 1.1079403162002563], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.3507473233305594, 0.34309058470100834, -1.162990160258063, -0.18708582428680418, -0.33946459566997783, -0.2276763234557067, 0.5968561605493233, -1.2790231397082756], [0.9666553294725934, -1.1279606882552755, -0.1884585268782212, 0.8869889091146606, 0.6636769044819368, -0.6910162887497601, 1.768866116135496, 0.36550956296160114], [-0.954442033734878, 0.042534448025081585, -0.8310523836388154, 0.4341813736815485, -1.3418177789944135, 0.48004707633804855, -1.7705279388998507, 1.0966492862222499], [0.27218775172401993, 1.3565218331371975, -0.7104634362171247, -2.094473651219682, -1.3695901507767492, -0.24259698253970766, 1.963839633884114, 0.5718240547581854], [1.311873622812302, -0.8777487448898099, -0.5316172746212495, -0.33294601985679023, 1.1693862959913428, -0.5647782400284748, -0.29091432631500325, 1.6209555583550392]], "dtype": "float64"}, {"__nd__": [[-0.8727419985794409, -0.032823359246225606, -1.3444572048872232, -0.23573173549933482, 0.8294296631594965, 1.1925441100986218, -1.391720049766009, -2.5252423765441576], [0.5386108678952128, 1.6672866723473339, -0.7007146526883685, 0.26455710030618845, 0.026768653880526846, -2.156833666400863, -2.6704244831860553, -0.06702310487501038], [0.4860854591358569, -1.5170509109211134, -0.3271258413868034, -0.7932327743848562, -0.9638483851375168, -1.2907776957995845, 1.170057498761513, 0.010588242986837262], [-0.23361905482565332, -0.48949991619971206, 0.3791281691397474, -0.5144317423686732, 0.1495012636619493, -0.7834517179750559, 0.33142353278050085, -0.3271003849287765], [0.5396368839884074, -1.5981056221909866, -0.8328156233745395, -1.2792349618540035, 1.5228744576907318, -0.6655056030601512, -0.12337524270476494, -1.7874720913003521], [-1.0222339033483543, -1.7855558570698475, -0.19477233152167991, -1.0585415426129554, 1.4393682671205945, -0.9793034965629355, 0.9051229215150605, -0.242434798330533], [0.4121873190479158, -1.0231148902740423, -0.15433535240959725, -0.2635465798900246, -0.8944220085129309, 0.44959552108222145, -1.599954045848526, -0.2784448507983786], [-0.30844204595846736, -0.016670769097965204, -1.2578335733715598, 0.5596928449458732, -1.0048215215092628, -0.7053324984781036, -1.0649845315325053, -0.4900336439474939], [0.2744451099768222, 2.8017277977663135, -2.7498647772831424, 0.8154776835749543, 0.7895258682071479, 1.3800259476693169, 0.1694307919115836, 0.3813503303896691], [1.713348785747476, 0.03182401158943564, 0.11498813900985208, -0.06658599354738985, 0.38615160425786166, 1.7165962169549192, -0.9455861695349351, 1.1198634689846245], [-2.3709282084908585, 0.414367480386855, -0.4757655811563425, 1.2254971612400698, 0.7545125053176746, 0.7329449804369732, 0.5165978908585432, 0.2771127775134856], [-2.088956731788394, 0.23528999726713878, -1.4213698110507533, 0.987672881567685, -0.7368325860777105, -0.06589160876950578, -0.5733697725316557, -0.0033968183904460876]], "dtype": "float64"}, {"__nd__": [[0.10754429325675272, 0.428303925209736, -1.3336414939064096, 1.4626208646196348, -1.1725799739492326, -0.27497081251353633, -2.806030143402162, 0.18548710174512717], [0.8629219515188047, 1.0944757898044037, 0.40740308145328735, 0.37095547839770643, -0.7773357832595578, -0.5194951171887373, -0.3166615175433137, 0.03238883717158658], [-1.1935833935299933, -0.4833492376762353, 0.4126020288159263, 0.047619783941804245, -0.24624980584509085, -0.9299261950582574, 0.17731577483609998, 0.07178317966009112], [1.3732423601176806, -0.09639131588725613, 0.8679225121291942, -1.4320804394629447, 1.6651726746076323, -1.4849039023039716, -0.20509159591389434, -0.49350773543258847], [-0.7238236018971211, -0.461579774602582, -0.68832431740873, -0.04900399839359427, 0.11447106536683918, 0.985808025572785, 1.0811660470463478, 1.6040251673447676], [0.32378267377434633, -0.9528829932594701, 2.1744907659043244, 0.4116487617287796, -2.676529005375767, 0.6222605906236327, -0.2821652832873429, -0.8500883029266798], [-0.7453168723213186, -0.0842275117794918, -0.9552628619532028, 0.09488082712964284, -0.31079791545835994, -0.31959766664408085, 0.3134463040134563, 0.1954988634834098], [1.4691158136021842, 0.3457563377701115, -1.080924007775614, -0.10110411269944829, 1.5788362531639453, 1.5909780961587576, -0.9103061633302669, 0.5634521454802971], [-0.5743572879343545, -0.3763301036233583, -1.645900725737879, -1.7387918572813117, -0.40867160885227766, 0.13384492519584382, 0.3836233863091301, -0.978285746743896], [-0.35147535141408254, 0.003274740787278153, -0.27786364642008815, -2.9069806145475026, -1.1750434457006758, -0.022836475702968966, 0.4505946070477635, -1.0491518484468454], [0.8947755091452118, -3.0933376746827466, 1.1269628332980208, -0.8513749312156031, -0.14054813303454863, -1.084458079987977, -1.3410299012597224, -0.46944276361360676], [-0.09939831453669669, -1.2865470669081636, 0.6448152811316662, -0.58529241964051, -0.06626589356790309, -0.8785498109712908, -0.8210919482495448, -0.19827320090302833]], "dtype": "float64"}, 4, false], "e": {"__tup__": [{"__nd__": [[-0.10615576058626175, -0.175186887383461, -0.49470147490501404, -0.3532694876194, -0.23674359917640686, 0.038588326424360275, -0.20732922852039337, 0.06619340181350708], [-0.1681087464094162, -0.6061451435089111, 0.5014437437057495, -0.30595263838768005, -0.5520014762878418, -0.16619838774204254, -0.013540143147110939, -0.09174086153507233], [0.22357633709907532, -0.3985907733440399, -0.14887011051177979, -0.4292413592338562, -0.1094161793589592, -0.22659170627593994, -0.5363383293151855, -0.11146428436040878], [-0.2928745150566101, -0.43630993366241455, -0.11124202609062195, -0.6674412488937378, -0.22551268339157104, -0.340288907289505, -0.00367339258082211, -0.2892276346683502], [-0.09827776998281479, -0.23976032435894012, 0.09221427887678146, -0.7683115601539612, -0.6232517957687378, -0.03142385184764862, 0.1016363576054573, -0.18836364150047302]], "dtype": "float32"}, {"__nd__": [2.909027576446533, 2.668044090270996, 3.308119773864746, 2.562685489654541, 2.7981162071228027], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.5335654743940903, 0.5142536130226086, -1.4534729507668458, 0.23714418106753812, -1.3135652152415505, -0.3943457610713792, -0.882746222878733, 1.0457577652694796], [1.538805980309548, -0.6022091974676057, 0.2822941307659521, 0.06876455990648171, 0.5765263409207497, 0.8890495017941767, 1.6683153787332983, -0.06700112251254604], [-1.9610074223597538, 0.5447688136363442, -0.533046253383237, -0.2966438218932699, 1.0966906262175324, 1.413919835222748, -0.03417992976886248, 1.1307651321200394], [0.32256261302127315, 0.6930091646193651, -0.24346213918338108, 1.2376000033515853, -0.715687352641389, -0.04936519618253392, 0.819766841917862, -1.082768805326475], [0.26396437482687113, 0.34522006286952023, 0.7115630223421053, 1.6259277384359083, -2.224277236144167, 0.8581961935469834, 0.2498144469144824, 1.0659331547926612]], "dtype": "float64"}, {"__nd__": [[0.5867823751196357, -0.7646924677717325, -0.663142980998386, -1.1074313542913765, 1.1302867000219725, -0.7337578793664978, 0.20802241797261062, -0.3510802626760997], [-2.176542582802754, -0.4581217921423171, -0.30933096892664147, 1.3250088028883715, -0.3543507515209771, -1.2662530235228528, -0.6880824895603563, 0.6953572184015436], [0.46955538661907403, 0.4242244220174294, 0.559009937950181, -0.9307619439631543, 0.806957097090844, 0.23532348748181528, -2.209290098598305, -0.7243509829952233], [1.2037161225255284, -0.6572645275587392, -0.4814746325769659, -0.059735925089861544, 0.05761553620798836, -0.11758609232703289, 0.4798593120573888, -0.842369177147925], [-1.557400667159789, -0.0955160554488326, -0.14069464105696614, 0.9176750302022483, 2.038846582117233, 0.4530054669617193, 0.8425855882792349, 1.6795968083038428], [0.7917072066945534, 0.6683715104431768, -0.9441356715768496, -0.41486980003818513, -0.7889856857723783, -0.3830807447913794, 1.0236919184172315, 0.6627653458427598], [0.0041682868635343024, -2.19613022621847, 0.5629652966077362, 0.5232590220614457, -1.1952835356768994, 0.053846116544323565, -0.3203211097900028, 0.611567102491786], [-2.1180738661115353, -1.3378258648286891, 1.2705178650834035, -1.102739108947013, -0.8640179252532476, -0.5107310018402003, 0.18578843545896095, 1.26266120044281], [-0.13601947026415803, 0.8763266729117537, 0.3882370798245415, -0.6122066850105151, 1.5934316172472758, 1.0051196080654246, -0.4017077153969582, -0.22250575453648375], [-1.0782569578880639, -1.7985874943385358, -0.17607220225555686, 1.0603910066158515, -2.965561317411082, -0.6631035870544183, 2.0673454139265286, 1.6239703123070077], [-0.9172621916512872, -0.6607594099903277, 1.4351305812931747, -1.0238750890681985, -0.00955181998779506, -0.8005083311764107, 0.5165025760074722, 0.27376111066229775], [1.8955721970894044, -0.9655761145353545, -1.587708326607266, -0.29691007207707887, 0.3428716220531728, -0.2619828267802848, 2.602368524351387, -0.4636481414549217], [-0.17854706316960117, 0.9859387481303149, 1.8138953852771775, 0.24529452918322223, 0.7463073073466431, -1.2249103964889874, -0.9423613774978422, 0.36982141847412825]], "dtype": "float64"}, {"__nd__": [[0.8535833145819416, 0.6373554345899036, 0.6088320076364142, 0.9073865179535819, 0.03958179687011248, -0.2578692831856559, -0.17964160078874394, -1.2373403978526003], [-0.6061371795287929, -0.7084250812511491, 0.725925848100176, 2.621832334705719, 0.8999840713501503, 2.1814567278057377, -0.7284113764909652, -0.2527036768626191], [0.005359936525535909, 1.382706796806993, 2.2538208715667634, 1.1638112721299583, 0.6878290660092324, 0.4375432855212948, -0.3515832927838017, 0.7477197479789911], [0.6987239216771278, 0.27759355271509784, -0.624964107460974, 0.3801674447125683, 0.12298698004431424, -0.5747179280725246, 1.4386463546755843, 0.14641904639417025], [0.8595550670363569, -0.9208225816149672, -0.20369631727399115, -0.506533652428212, -0.8575351130046348, 0.351554048537489, -0.005875205071845447, -0.9921763066148861], [-0.4351523570898661, 0.282702470485965, 0.1943684058608488, 2.114103843758606, -2.1049013265184846, 1.1294304933239414, 0.9905352670154041, 0.7153956420817812], [-0.48553811820444137, -0.494098387240886, -1.7396907803432649, 0.7193936051316842, -0.34029144320735505, -0.33049331305052415, 1.900189738121709, -0.7293475128113953], [0.19200674094204664, 0.8976161696121265, 0.13975329459483446, 0.6710427702551948, 0.5150633409904183, 0.5345064195517635, -0.9624478091619499, -0.9939577529181364], [1.3746187238431673, 0.7880780157052958, 0.08706252595549704, -0.4227708970732933, 1.718189532906842, -0.08390388898756114, -0.7010189023796407, 1.308160438185032], [-0.7845310385806001, -0.10510693154461048, 0.3823723677870512, -0.8747982587482743, 0.5301006855820048, 0.5553245416791827, 0.7888507236872296, 0.07453607624956742], [1.8480186294827177, 0.242384208618892, -0.17444619724208452, 0.6340554117859104, -0.20791521710239153, -2.1740617224471146, 1.4840419667552394, -1.4963784286948616], [-1.687767672392959, 0.6120122111052694, -0.4101468218030139, 1.3569083071699646, 1.2845898507775468, 1.357995977559167, 0.03500272842289303, 0.5875384605741423], [-0.993188581213568, 0.9815381577988909, 0.1644557246919225, 0.11179800840891718, 0.3752740327103277, 0.30447308157842506, 0.6488347273824, 1.0462615570997444]], "dtype": "float64"}, 4, false], "e": {"__tup__": [{"__nd__": [[-0.33263733983039856, -0.12927351891994476, 0.3272688686847687, 0.9678941369056702, 0.364041805267334, 0.9785420894622803, 0.05035335570573807, -0.2076137363910675], [-0.6739550232887268, 0.42038238048553467, -0.2468964159488678, 0.9329374432563782, 0.6109185218811035, 0.6728469133377075, 0.33575382828712463, 0.23341114819049835], [0.5911354422569275, -0.2354990839958191, 0.040410369634628296, 0.11136309802532196, -0.09425618499517441, 0.36875200271606445, -0.10184065997600555, -0.46456068754196167], [-0.3208465576171875, 0.24499891698360443, -0.011433075182139874, 0.739172101020813, 0.22520482540130615, 0.5090088844299316, 0.5037582516670227, 0.14195777475833893], [-0.5776597261428833, -0.06507379561662674, 0.11805526167154312, -0.21780310571193695, 0.30302318930625916, 0.4898545444011688, 0.7485124468803406, -0.04274650663137436]], "dtype": "float32"}, {"__nd__": [3.249570369720459, 3.2763757705688477, 3.3611254692077637, 2.506453514099121, 3.5900580883026123], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.14864404484178556, -0.2303737089687008, -1.0996358191742002, 1.854480305590387], [0.563163292904423, -0.14223869207214868, -1.4026251569666437, 0.7232745089552551], [-1.1204409048828072, 0.41957737093621444, -0.2407880251703682, 1.2396489294372934], [-0.2991656093533991, -0.9865539393913091, -1.9595015194551595, -0.44551574559332247], [1.7233754138812192, 0.37552190842536076, -0.31201967481258647, 1.2384979246478118], [-0.5447025288006724, 1.064032029767179, -1.2092045545740115, -1.103948037761054]], "dtype": "float64"}, {"__nd__": [[0.9288602710787215, -1.2799001038191051, 1.1376094723653776, 0.044344122357689496], [1.9291961759263585, -0.50676271983615, -0.2052186433703958, 1.6274082113881907], [-0.5728337959607628, 0.2642543401778171, -0.8520683997005555, 1.017039906714969], [0.07587826128817358, -0.9847491180351889, 0.12237807115454684, -0.9100182861344523], [0.7905047022961387, -1.0619035749217802, 2.4394962394673843, 2.3300692951843764], [-1.7418009944139559, 0.5384175917132369, -0.7596231295401269, -0.680993316570695]], "dtype": "float64"}, {"__nd__": [[-1.1614609625710137, 1.5758110954998152, -1.79131607378969, 1.7250224483098788], [-1.5643955239787093, 0.9415094141830559, -0.07670936482844325, -0.4617996631174593], [-0.8524465833336431, -0.27641182609300236, 0.39380316149415223, 1.637890829399593], [-1.740404951690721, 0.4003977251469909, -0.09848446974417986, -0.6818325295706889], [-0.06959384737885983, 0.046384485872541965, -0.795768974181604, -0.4364413965947043], [0.37279686424609393, -0.3789388062958146, 0.4729114042431385, -0.8069338926570797]], "dtype": "float64"}, 2, true], "e": {"__tup__": [{"__nd__": [[-1.1614609956741333, 1.5758111476898193, -1.791316032409668, 1.7250224351882935], [-1.504279613494873, 1.0361440181732178, -0.33252018690109253, -0.13553662598133087], [-1.0238127708435059, 0.14218086004257202, 0.09281855821609497, 1.2271223068237305], [-1.325828194618225, 0.3753809630870819, -0.10198275744915009, 0.45872408151626587], [-1.171220064163208, 0.7019175887107849, -0.3225598633289337, -0.17535151541233063], [-0.180014505982399, -0.18132567405700684, 0.31398147344589233, -0.2998315989971161]], "dtype": "float32"}, {"__nd__": [-0.36789801716804504, 1.4732931852340698, 1.4579384326934814, 1.579592227935791, 3.073613166809082, 2.0199484825134277], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.3333621317791501, 0.5456393671207658, 0.5114661211266899, 0.3603325934542445], [0.12443611590391371, -0.10859705878596869, -0.5558462757446445, -0.5300627886379878], [0.2086633069392911, -1.367952811203102, 1.6853357721600526, -1.2602559404210827]], "dtype": "float64"}, {"__nd__": [[-0.04345015880177048, 1.336079529292427, 1.8036381603855993, 0.7827867888534936], [0.21217750509669953, -1.2429609775141508, 0.5349745958869535, -0.1553016444171877], [-1.2307992054282408, -0.1671119958997985, 1.1680062917377119, 0.7748469185441446], [0.5934201524670298, 1.0461467743745336, 0.7265800560355407, 0.5419453035474413], [-0.0337553865530589, 0.3457812368990469, -0.2130665076818474, -0.4114162670969542], [-0.6164513374495892, 1.7909070062023615, 0.7751137991994781, 0.9985875681688042], [0.20156423723286435, 0.4301354659881542, -0.4793273416569439, -0.3820552368900358], [1.0754179210255528, 2.0091597600429583, -1.6878908145319937, -0.6121085552496288], [-0.6779292450079406, -1.120680485037113, -0.5378761787766404, -0.4643967576333777], [-1.0556830602030982, 0.3141314193607263, -1.6293815079318374, 1.354543763198212]], "dtype": "float64"}, {"__nd__": [[0.7406597323617218, -0.3804792125310222, -1.0137034643302638, -0.9515698205173051], [0.06372275934727596, 1.6165684730104302, 1.1718952411114287, 0.9495968741853802], [-1.0239681648897936, -0.029115225296439486, 0.7355977587977682, 0.4305575317617178], [0.39372272756259474, -0.09637604516387879, -0.28452653672603756, 0.02427720483202018], [0.2778756211641756, 0.04188324039978785, 1.1863599342545417, -0.5747591787065114], [2.1593961427266226, 0.6318814207611722, -0.02690714619608754, 1.01866253678106], [0.640451058498218, -1.4981290853513793, -0.1978343133829671, 1.3214185652700585], [0.3289177835592895, 1.5886211729728807, -0.4564979138805301, 0.13429314463261738], [-0.324539633575802, 0.14275981203548327, -0.5352186188847556, -0.5609240728665607], [1.2353715202910447, -0.9653007599175748, -0.19744699142390545, 1.0459523469768746]], "dtype": "float64"}, 3, true], "e": {"__tup__": [{"__nd__": [[0.6265895962715149, 0.16729788482189178, -0.08487079292535782, 0.1626552790403366], [0.2918911278247833, 0.32950153946876526, 0.052385736256837845, 0.2022002637386322], [0.0476556271314621, 0.5106632709503174, 0.4116801917552948, 0.3007034361362457]], "dtype": "float32"}, {"__nd__": [2.473290205001831, 2.1707026958465576, 2.4179577827453613], "dtype": "float32"}]}}, {"a": [{"__nd__": [[16.869933427245115, 13.695751820114436, 7.53876138956839, -0.9110846122297699], [1.3052775269518817, 8.066216988022, 0.8113042793780726, -12.55244851913734], [1.596713675206391, -1.469759903197039, -20.72280481695572, -0.3323816401253177], [-3.5846101701301025, 3.5738233856484642, 1.4775861757868687, 10.939809449042004]], "dtype": "float64"}, {"__nd__": [[10.412867407910872, -2.9313922287581993, -10.00737058278144, 0.512966823577741], [12.109115601052691, 0.5745609195912053, 1.573620003716042, 4.42999751011515], [-0.5489570902984405, -0.9512543597144536, -0.5293292755574214, -0.1430418421542957], [-9.045575348487152, 13.779361382473136, -7.416419264729158, 8.458824044852772], [-6.0347838884082465, 8.580337273450832, 1.8667860515101358, 4.635064165791803], [15.918868470368526, 4.622910949298949, -5.329865484509836, -4.500725830141707], [-12.505387456128592, -10.340325911963635, -6.9971455810506455, 6.634772034653072], [-3.013651506071742, 5.18023735323848, -5.860852166285412, -7.844550549279077], [-3.4959974928732684, -6.038901343826656, -0.225646700136938, 16.885377048161388]], "dtype": "float64"}, {"__nd__": [[0.16473982448629698, 0.9468133806734473, 0.1866318887199723, -1.0185768306074063], [-1.1272298423123555, 0.8613974428594067, -0.18959054190478963, 0.510617326413531], [-0.7276751858329389, -1.3981081964830653, 0.40626891736533666, -0.03809768205106909], [-1.4234379914612103, 0.4963352591424826, 1.3648387894851395, -0.1022310734012751], [0.18548946153486837, -0.4611282658038174, -0.36293407231448416, 0.4212476420935357], [1.166984107414436, -0.16698750240809318, -0.3794858590933018, -0.8579821839655488], [0.16000361461756304, 0.5764300988114371, -0.2427303499012556, -1.275289071728332], [0.9798622967235935, 0.6841080644103897, -0.6946857286704391, 1.451305129538392], [-0.0010076809605761951, 0.09995104563936948, -0.7397394203877059, 0.8405727495575146]], "dtype": "float64"}, 16, true], "e": {"__tup__": [{"__nd__": [[1.1669840812683105, -0.16698750853538513, -0.3794858455657959, -0.8579821586608887], [1.1669840812683105, -0.16698750853538513, -0.3794858455657959, -0.8579821586608887], [0.16473981738090515, 0.9468134045600891, 0.18663188815116882, -1.0185768604278564], [-0.0043707056902348995, 0.1008882075548172, -0.7347636222839355, 0.8383436799049377]], "dtype": "float32"}, {"__nd__": [147.89224243164062, 55.11946487426758, 114.07255554199219, 87.67198181152344], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.9740861551524708, -1.1801265386944684, -0.0430223195035832, -2.055891301163295, -1.3964938967027274, -0.7161299806215503, 0.7524777445296665, 0.1266338604062119, 0.39353323208881863, -0.4957187441650112, 0.5362918170009964, -1.2550293184568002, 0.630772032836894, -1.0766267164494123, 0.33675936023813, 1.4427153824891523], [0.9212969627103992, 0.9671052794974626, 0.698113765770949, -2.3645100931173153, -0.6780967388392963, 0.8022859246451792, -0.11256568466264999, -0.6978779305369722, 0.5357160130500712, 1.162333511132872, -0.6410622353695855, 0.6325425484294717, 0.31469534184775577, 0.7000302330217031, 1.7212547853767854, -0.049100447137419546], [1.1153541086198666, 1.8292695848853033, -1.28419454583061, 0.07664500221142526, -0.4162756557286444, 0.6991947446918731, -0.5656880615177582, -0.18120244543826092, 0.8435917678446704, -0.0003746562651581894, -1.0536766539557012, -0.5871910968206672, -0.7235222378561302, -1.111142486092898, 0.9189569848032076, 1.5647181128565815], [-0.9094429623920128, 2.207378610221001, 0.5616267320074906, -1.625979928429215, -1.6779386804568066, -1.1501995004235503, 1.1405731295543093, -1.476722755076865, -1.0664362555040063, 1.2896416615841093, -0.4598971141869625, 0.2516352528530253, -1.7813192588312432, 0.0694507090919959, -0.9924199119928335, -0.8126093230365635], [-1.396522368514345, 0.3117678899375947, -3.1063773952078346, 0.0846806679141445, 1.141324353821361, 0.8610598380899122, -1.7759260544171085, 0.7849522863850473, 0.5503861758473586, 0.8966986386516442, -0.593836580529804, 1.3092540386567522, 0.5044212021505043, -1.1550330013170278, -0.7865628854698736, -0.3054126550597854], [0.706965360139849, 0.6984754719741112, 0.714324557718387, -0.8146550587951454, -0.2062602251383572, -1.080626976839402, -0.0973517825658157, -1.1192149684118955, -0.25068817698831963, 0.9837863279164241, -0.6151630194679696, 0.97170026994695, 1.3367149273108936, -0.8941533763556131, 1.9130570606377477, 1.0912558240061236], [-0.7008148779876363, -0.12259987699664429, -0.6763891385859736, 0.3864529364092026, 0.4742108455869937, -0.4758251151390759, -0.05168332262734471, -0.48823101842493544, -1.6619817490558875, -1.476323450269063, 0.6619931946475379, -1.8423716731008117, -0.3665623430696784, 1.234282699271971, 0.8041413047631614, 1.3431769186258653]], "dtype": "float64"}, {"__nd__": [[-1.3942525708712556, 0.6717242103621468, 0.6475587184868975, -0.07724194568442369, 0.34914709776856767, 0.03703919739271449, 1.6736802343814188, -0.5944072499625536, -1.2490542706304724, 0.9980731304965973, -1.3605202171756026, -0.8502946962525065, 0.33824853261584054, -0.4760170168697923, -0.9789288158372265, 0.9288236071525225], [-1.8566519492629578, 0.3169484027554536, -0.6929646571738286, -0.3900387689034653, 0.6779984604571657, 0.48459423601840096, 0.13745480401826637, 2.0305426660201524, -0.6081227183022359, 1.7221569838105124, 0.2177441902335987, 0.08796915051639978, 0.29821309256891365, 1.1307402040842383, -1.6572202171580392, 2.1207926598424938], [2.258422984921996, -1.2184234876929703, -1.2399566804280107, -0.4694769172233329, -0.46248309084095746, 0.013569717744074538, -0.02821928819369128, -0.09415410778076815, 2.1110127827733063, -0.6477543613674726, -0.9111261690775605, 1.1144941551839818, 1.4458589556815575, -0.584931251246554, 0.4336291102615471, -0.08873969216519512], [-1.5648889590529418, -0.4018369710434743, 0.4863956627342323, -1.3274426780274873, -1.0651892909535157, -0.10283784316938269, 2.3270949706054815, 1.4476250197563751, 0.12130364597732986, 0.26328512959040934, 1.3071803617462414, -0.3934706165179813, 0.543151221058522, -0.7004286453186188, 0.38380113758160384, 1.0985654802448876], [-0.9631448249511881, 0.8452705871014181, 0.9580620600465548, -0.29686198623851867, -1.8351677973381753, -0.017065590776069523, 2.831685857346476, -0.25650830949835907, -0.9263249594766328, 0.6813981389708913, -0.55492871976431, -0.964830401711979, 0.39309828720891277, 1.3958120355468127, 0.3695173947301892, -0.04285583040980598], [0.3437593598671674, -1.0876084859696704, -1.1899403618281061, 2.0892550511137387, -0.2176367190815979, -0.10792868974141262, -0.17410722278245247, -0.9290473514200228, 0.7662100674321115, -2.618731989298956, 0.3641895757081908, -0.5888113410084975, 0.3208717977341458, 0.913285482474821, 0.1946705390383418, -1.2622667959539047], [0.6090332995286553, 0.1313425462622658, 1.7970410104231964, -0.813981152696833, 1.9740175166054323, -0.6143563942880778, -0.868307583304617, -2.7352187900986418, -0.410003984072958, 0.7194038046050713, -0.41046124911114684, -0.23187182013507776, 0.9359641023148058, 1.022179307136895, -0.674627347180856, 0.9333383998663454], [1.0254405078382216, 2.6793321451568595, -0.7523539690618507, -1.743425263449859, 0.17679573909961394, -1.0625914645094068, -1.0116007405385916, 0.06229935963167987, 0.25336374993591, -0.16847064411726492, 0.48352461750687814, 0.5088240943652215, -0.32919808585148524, 1.173646697104018, 1.6294993442785417, 0.15362761837611993], [1.3822209336745956, 0.46515106016221186, -0.17183737329015297, -0.22161102238524816, 2.134297183826976, -0.8812594717269225, -1.1271532300832643, 1.222068176007562, 0.09470690239739034, -1.5257665471363453, -1.1805122385961488, -0.2825258668433602, -1.2832894956540937, -0.21612943023773135, -0.3180953481363407, 0.22572834821035656], [-1.3624309159337165, -0.18570534630801974, 0.24691906826129217, -1.498455883710796, -1.4984832857206558, 1.2236647828276601, -0.35969108520668, -1.3237245042897374, 0.988208495664122, -0.28954457195259836, -1.7269283499850059, -0.5250499197045159, 0.18793515849536283, 1.2168188707158156, -1.2742647008317087, 0.2642108379513048], [1.077746399064635, -2.806763174582491, -0.16663340814511035, 0.6239449540789018, 0.01640948442956724, -0.23891157443484085, -1.227204764063533, 0.48089188046021225, -0.18883557739665663, -1.0356401846374472, 0.4709439023315973, 0.5122047597200344, -0.3638733159028714, 0.5518856840166427, 0.04330018144940791, -0.10553935001747052], [2.685298928479472, 0.09426065029784092, -0.47340561872084785, 0.5463053265344237, 0.9864490246878104, 0.1145895901692593, -1.4553476461418815, 0.242869948819979, -0.38868579567211436, -0.46890395837662063, 1.2266665293698469, 1.1429711294428961, 0.6520185025245797, 1.6270945003897626, 1.0273057489903261, 1.264561115614597], [-0.06756364767470228, 0.33991402505253965, 0.08466867602944465, 1.9497678699383194, -0.37415997515935867, 0.9271953159442741, -1.0139828897996026, 1.0732900402700465, -0.3203536720798256, -0.738251507719868, 0.8810583853175966, 0.6363963787533319, 0.008347437743591798, -0.30936009520998037, -0.052830958680666604, 1.2907211893917419], [-0.24881774346148972, -0.9768766647115563, 0.9760975724628536, -1.4093082931159049, 0.08557309383177078, 1.123797712450753, 0.15357312666527642, 1.3562498516458004, 2.443406591680171, -0.8519386560576019, -0.906509226741936, -0.5493277125808204, -0.48032051591145053, -1.3705589383460508, 1.3843371336957606, 1.0762842457058424], [1.4755568304755524, 0.18020613381425676, -0.14442943332814165, -0.7667481684445384, 1.1332963347429472, 0.783678618255615, 0.5392875996484405, 0.13757909822570524, -0.8581523689708707, 1.3432405995408736, 1.3634253365226812, -0.18080244725861602, 0.09744122600539779, 0.20548065822942535, 0.1746280980260215, 1.6532397847005162], [0.5050225807970453, 0.030795168149224755, 0.6908116181979157, -0.06021206307831176, -0.7002838770788115, -0.24578055027070655, -0.6611138782449575, -0.3745339653240921, 0.7280836914789732, 0.5978705838248802, 1.5855773385454397, 1.1737645202006395, 0.4912446138251638, 0.39013709596524276, 0.4148055890774431, 0.09488710438346926], [-0.46088773645592285, 0.7134754608174263, 1.961904977218242, 1.1031804523000508, -0.8333395576747045, -1.600061785155835, 1.0574617350729112, 0.2523642360309741, -2.024108354592823, -1.9697293637171893, -0.492737616012614, -0.9595605669351417, 0.24513288431175656, 1.060510633124899, -1.1708861559895016, -0.000605317970791789], [1.3846339556463725, 0.5479497795733993, -0.3058116260826324, -1.2430796370494133, 1.3172610157878264, -0.5849606229283721, 0.9551176508546024, 1.8068961325079833, -0.698288543874307, 1.838766054991641, 0.32225573345457703, -0.6326286097042839, 2.833220158199736, 0.10080007703775712, 1.222404161277782, -1.53110702700983], [0.358859086343144, -0.4535773881246974, -1.0381230365509533, -1.2414041648894654, 0.42624489826936035, -0.7763446390407036, 0.7550492730543824, 0.44230478324808753, -0.8426696053403128, -1.2509505391503861, 0.41138362221293256, -0.8471663917354799, 0.4759641843592693, -0.8839699347792705, 1.2600754080126215, -0.07559248106400568], [-1.365032225985973, -0.17393206862332858, -0.5416811166990975, 0.7295022710938741, 0.418720871350498, -0.30924759648487427, 0.726485024292873, 0.8873992847991213, -1.5722823904694128, -0.9226930195442187, 1.716852772845836, 1.1405080203840323, -0.724696535954661, -0.2127091837797916, 0.8765623907704044, -1.9580215264787102]], "dtype": "float64"}, {"__nd__": [[-0.2892664468077736, -1.5831485712579776, -0.36292336950205445, 1.031948893954407, 3.043902825237671, 0.5869751989365406, 1.5963127713753094, -0.7281121088479878, 0.11585794593880859, 0.6667186481641807, -2.17071302552264, -0.6358533018000193, 0.9387612828844666, 0.3381560065609014, -0.8929352062570534, 0.5973808749458267], [-0.6189775852140852, 0.395175554450311, -1.3133205697039387, -0.4626119767746675, 0.4222399457562474, 1.1941315237256962, 1.9641016478060036, -1.3354762189614138, 1.3302565933648862, 0.8522274579353203, 1.530076859235956, -0.03649914293778985, -0.503542032329127, 2.348631179290809, 0.29706153561978366, 0.842510510647973], [1.1240813541821113, -0.5695573959354608, -1.929231746174113, -1.036245566635322, -1.8604806634980666, 0.39539536169003503, 0.9245651728044789, 1.7101040204421893, 1.520213134581839, 0.9791524296358611, -0.6778886757209277, -1.2927318014381524, -1.3917109410801674, 1.0670831597457302, 0.04688805806742035, 1.1508984063493313], [0.9219537973076781, -1.3694360597809145, 0.8186092435707616, -1.3118073458048334, 0.5176227076257001, 3.0004645099130687, -1.363164141087513, 0.48376190955336357, 1.8744910826211014, -1.5842524888892537, -0.9096623754307304, -0.902175477193542, 0.1638616130963518, 0.7000806967115207, -0.3419331907740446, 0.3195051930110127], [-1.2538391599729843, -1.0174243921122164, -1.7641531187255872, 0.5689994071082252, 1.637934594704494, -0.3480524685789175, -0.32689951223850316, 0.4956243579342946, 0.6140088153641702, 0.55044291518029, 0.8062497665301939, -1.661006782301087, 1.5130997323259778, 0.07445019659540117, -0.06513275411990727, 0.0735127293651855], [1.5010350598267836, -0.661121352962415, 0.7410724449643946, -0.3798634324436877, -1.0161531802092436, 0.211577477843554, 0.9094601582790953, 1.4605421926459339, 0.2758560811451864, -0.195608334806432, 0.502767718562805, -0.1263035397602682, -0.48907671062111163, 0.6601660591646445, 0.4905035806177188, 0.2824793978318956], [1.5815212248844221, 0.001457204234809118, -0.2154099835641696, -0.1893362656658029, -1.1529985990436167, 0.09705652960701365, -1.9843171967465916, 0.335596954773087, 0.45767209950301674, -0.923068206756826, -1.481171710344781, -0.3932203370830844, 2.3381285725568586, 0.1314154141693266, 0.2568896514132355, 0.34972485967024686], [-1.314834111982029, -0.10390064697067987, 0.372984049187199, -0.5391464714835945, 1.0990012567270893, -0.5125404022324854, -0.16495916045811715, 0.9995185592225249, -0.5869262037857004, -0.3324215857655456, -0.3270229914936275, 0.05087525564622946, 0.3794761887538131, -1.1521239108330044, 0.737865822010124, -0.19906400186624504], [2.0083428360522646, 1.8143317803787846, -0.9616378368633445, -0.6923430421385594, -0.8669876073456518, 0.1738869010481117, 0.1667389152011375, -0.865076963851745, -0.3389541039728644, -0.2049317321206679, 0.4470164791137063, -1.2618596563403996, -0.12617340344928488, -0.8021843115247691, 1.0176405125586712, 0.9798650932980264], [-0.39118863639783846, 0.26102269618171, -0.7129035441546249, 0.19572892571176798, 1.396950863994733, 0.8379642895123309, 1.1199229638189931, -0.0009342433315272432, -1.9000737640232888, -0.4473800082987087, -0.37575591688028254, 0.7330313439145982, -1.5445697824112194, -1.6683014832890297, -0.8255455632767514, -0.2882802452536508], [-1.5458189427505888, -0.9046949419168707, 0.4698033718163387, 1.119145951616998, -0.2840043574342708, 0.5208225562589863, 2.1206221279650115, -0.9117243672585049, 1.3198083477635092, 0.5716128276597289, -1.3849136947502618, 0.14667858607759146, 0.775257093453481, -0.49751442468217166, 0.026262048609038965, 1.8960843419019655], [0.37567317670375006, -0.8863237406755319, 1.032285822466939, 0.834239179537504, 2.0004877220435047, -0.8767453780418054, -0.3893642918646705, 0.776828375719607, 1.5767793831328132, -2.4672244416757256, 0.2517513410967149, -0.986199854098766, -1.6981595747171228, -0.33272734450564906, -1.050392684936262, 0.017626494732984603], [-0.17925596939105332, 0.8917688928415684, 0.6876029314862131, 0.013768745364962753, 0.4069480707738493, 0.8514578870968388, -0.01869364862501997, 1.8830300367377308, -0.40414286737698685, -1.0837161487130826, 0.38310677236730645, 0.3666955582715291, 0.8276293577358405, -0.4856074531664522, 1.5807677368837074, 0.5960931813872875], [0.514220559913896, 0.434757950070447, -0.6384454083508775, 0.4696636633853042, -1.0459933186382817, -0.09488731607599078, 0.26387565593292517, 0.3414900466293046, -0.47669868533672105, 1.235464727675483, -0.4744697896805018, 0.32665126993181126, 0.11217729186673259, 1.4635569960706882, -0.06807855833943413, -0.17593314596258086], [1.2555162878145711, -0.2028845112935876, 1.3875857914944425, -0.07961405719397524, -1.0901283504342811, 0.5210715602014562, 0.3033447101770899, -0.04461458593257739, 0.15471966793168354, -2.0231704713012486, 1.289766054396596, 0.4818680953384973, 0.7542679275845318, -0.5554723736286625, 0.7183299105753735, 2.109483508717216], [-1.5691853112921625, 1.5899643115036999, -1.501580632546545, -0.012002685651689016, -0.6717578031882122, -1.54754261268384, 1.1031646635385706, -0.20376327872595326, -0.5813509978570583, -1.2533573130458178, 0.1696423079148399, 0.43756682018805015, 0.46206871518502124, -0.941443852371789, -0.6655499940952032, -0.062564012122629], [-0.5725715710482635, 0.40205331177667075, 0.20456519769235731, -0.8522067920384974, -1.7289111749924828, 0.33885902443592936, -1.0704141398679463, -0.01737043518351349, -0.3893640578574889, -1.3118194293179974, 0.9002473201399814, -0.6477878069656098, 0.7699916599510918, 1.0948011470339531, -0.10567707389175875, 0.5884063860585903], [-0.07717557878210336, -1.0728058635569504, -0.42200405511309874, 0.6347970341733831, 1.4493678918799597, -0.6077050550977353, -1.814806382240287, -1.6886168828346653, 0.6600082248255076, 0.6661306369905756, -0.15999409084528557, -0.6412963514250313, 0.20481692035735766, 1.0766883743241487, 0.19655851274714584, 0.4913299645346814], [-0.897998086198262, -0.5062910398092467, 0.370312491408965, -0.7078035313655807, 0.9857783107911962, 0.10081901939698355, 1.7820429991576596, -1.2847029137445505, -1.5413520575035067, -0.35026611790153794, 0.2451604508523145, 0.6416624896617409, 0.3688923238023937, 0.5875438117107874, 0.7631194054468347, -1.0553599693925064], [0.9505635141720248, 0.7873400367742255, 0.3260622707372068, -0.3108177101759488, -0.18308740350897898, -0.08831305703790797, 0.4762736580403668, -0.8044336603199365, 0.09706879476151559, -0.2953858677178469, 0.6388735889942314, 0.28150600602456144, 0.7253247601356221, -0.2683414611088697, -0.3784798530687797, 1.2428127209741715]], "dtype": "float64"}, 1, true], "e": {"__tup__": [{"__nd__": [[0.4414885938167572, -0.5051578879356384, -0.2851482033729553, -0.3477775752544403, -0.07203151285648346, 1.0071548223495483, -0.04709748551249504, 0.4997290372848511, 0.7262942790985107, -0.105079784989357, -0.5634544491767883, -0.5071877241134644, -0.03444021940231323, 0.5925353765487671, -0.1173473671078682, 0.3653327524662018], [-0.15331600606441498, -0.2167540341615677, -0.1698288470506668, -0.15593062341213226, 0.40030890703201294, 0.049950774759054184, -0.013435516506433487, 0.587870717048645, -0.02343820594251156, -0.3151152431964874, -0.2656705677509308, -0.23706065118312836, 0.1960277259349823, -0.3208528757095337, 0.16823837161064148, 0.23801963031291962], [0.20219190418720245, 0.10137779265642166, -0.21969260275363922, -0.1548680067062378, 0.04061174392700195, 0.08365276455879211, 0.2664165794849396, 0.5116549134254456, 0.056349486112594604, -0.2526511251926422, -0.1059989184141159, -0.27409276366233826, -0.03777386620640755, -0.09948819130659103, 0.2801894545555115, 0.4255467355251312], [-0.6499912738800049, -0.6094021797180176, -0.788628876209259, 0.204568013548851, 1.1594181060791016, 0.17852197587490082, 0.08206666260957718, 0.16216102242469788, 0.09966396540403366, -0.014438237063586712, -0.06728599965572357, -0.7814752459526062, 0.7946459650993347, -0.015337682329118252, -0.19251322746276855, 0.22720104455947876], [0.004488912411034107, 0.13321194052696228, -0.6394977569580078, -0.21505114436149597, 0.13634595274925232, 0.5296229124069214, 0.8414577841758728, -0.17768414318561554, 0.5833967924118042, 0.13705143332481384, 0.37595227360725403, -0.24984398484230042, -0.2430686205625534, 0.7418985366821289, 0.2748577296733856, 0.6875008940696716], [0.0752381756901741, -0.28039368987083435, -0.22429366409778595, -0.06418830156326294, 0.24774959683418274, -0.06544318795204163, -0.2594161331653595, 0.10893816500902176, 0.2555747330188751, -0.41326701641082764, -0.3208869397640228, -0.3419489860534668, 0.4240768551826477, 0.07447631657123566, 0.1097790002822876, 0.3492310345172882], [0.03544527664780617, -0.12719133496284485, 0.11358305811882019, -0.21929076313972473, -0.020077064633369446, 0.2597945034503937, 0.20771464705467224, 0.032745856791734695, 0.061027996242046356, -0.5996620655059814, 0.21442502737045288, -0.27945902943611145, 0.3519228994846344, 0.27370333671569824, 0.14955762028694153, 0.43616124987602234]], "dtype": "float32"}, {"__nd__": [3.468423843383789, 3.536430835723877, 3.2536988258361816, 3.800797700881958, 3.1145706176757812, 3.5532853603363037, 3.5935535430908203], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(blockwise_attention, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")